# MACE 开发实战指南

## 1. 模型概览 (Model Introduction)

**MACE (Multi-Atomic Cluster Expansion)** 是一种先进的主流机器学习力场模型，它结合了高阶等变消息传递神经网络（Equivariant MP-GNNs）与原子簇展开（ACE）的数学严谨性，具有以下核心特性：

* **高精度预测**：能够以接近量子化学（DFT/xTB）的精度，预测材料的能量（Energy）、原子受力（Forces）及维里应力（Virial Stress）。
* **等变性架构**：通过处理原子间的旋转、平移等变性，模型具备极强的物理表达能力。
* **广泛适用性**：既适用于小分子的动力学模拟，也能高效处理晶体材料的大尺度模拟任务，是材料设计与筛选的有力工具。

---



## 2. 环境构建 (Environment Setup)

> **💡 平台优势提示**
> **OneScience 平台**已提供**开箱即用**的全栈预置镜像。
> * ✅ 用户**无需**手动进行繁琐的环境安装。
> * ✅ 启动即可实现模型的快速开发与部署。
>
> *（以下内容仅供高级用户了解底层环境构建原理及依赖关系）*

### （1）底层构建流程
本项目的运行环境基于 **Python 3.10** 构建，并针对 **海光 DCU (Hygon DCU)** 硬件进行了深度加速优化。核心构建步骤如下：

1.  **加载 DTK 计算环境**
    通过集群 Module 系统加载适配的 DTK (Developer Tool Kit) 环境，为 GPU 加速提供底层驱动支持。

2.  **配置 PyTorch (DCU 版)**
    从光合开发者社区获取专为 DCU 优化的 PyTorch 安装包 (`.whl`)，确保深度学习框架与硬件指令集的完美兼容。

3.  **集成 OneScience 服务**
    基于源码安装 OneScience 平台核心服务组件，打通数据流与模型接口。

### （2）构建脚本示例

In [ ]:
# 1. 在conda base环境，创建并激活隔离的 Python 3.10 环境
conda create -n onescience_matchem python=3.10 -y
conda activate onescience_matchem

# 2. 加载底层编译器与 DTK 环境 (根据集群实际 module 名调整)
module load compiler/dtk/25.04.2

# 3. 安装适配 DCU 的 PyTorch 全家桶 (DAS 1.7 / DTK 25.04 / Python3.10)
# 安装核心库 Pytorch、torchvision
pip install https://download.sourcefind.cn:65024/directlink/4/pytorch/DAS1.7/torch-2.5.1+das.opt1.dtk25042-cp310-cp310-manylinux_2_28_x86_64.whl
pip install https://download.sourcefind.cn:65024/directlink/4/vision/DAS1.7/torchvision-0.20.1+das.opt1.dtk25042-cp310-cp310-manylinux_2_28_x86_64.whl

# 4. 源码安装 OneScience 框架及领域环境
git clone https://gitee.com/hpccube/onescience.git
cd onescience
pip install -c constraints.txt .[chem]

## 3. MACE模型构建
### 项目结构与功能介绍
本项目采用“输入 (Data) → 计算 (Train) → 产出 (Outputs)” 的标准工程化结构，旨在实现数据管理、模型训练与实验记录的解耦。具体目录树如下所示：

#### 1. 💾 数据层 (data/)
* **定位**：存放模型训练与测试的原始材料。
* **内容**：通常为基于量子力学（DFT 或 xTB）计算得到的原子构型文件（`.xyz`），包含能量、受力及应力标签。
* **获取方式**：
    * 用户自行上传准备。
    * **OneScience 官方源**：平台提供若干典型数据集，可通过应用商城搜索“OneScience”，点击“数据、材料计算”下载。

#### 2. ⚙️ 执行层 (train.py,scripts/)
该层级包含核心算法与辅助工具，是模型开发的“控制中枢”。

* **`train.py` (核心程序)**
    * 基于 **OneScience** 平台与 **PyTorch** 框架开发。
    * **职责**：负责组装 MACE 模型、加载数据、定义损失函数，并执行反向传播以更新权重。它读取 `data/` 并将结果写入 `outputs/`。
* **`scripts/` (工程工具箱)**
    * **启动脚本** (`run_train.sh`)：管理超参数配置（如截断半径 `r_max`、批量大小 `batch_size` 等）。
    * **后处理脚本**：包含模型转换工具，用于衔接下游应用。

#### 3. 📦 产出层 (outputs/)
采用**实验名隔离**的子文件夹结构，存放所有训练产物。

| 目录/文件 | 功能描述 |
| :--- | :--- |
| **`checkpoints/`** | **断点续传**。存放训练过程中的权重备份，支持意外中断后的训练重启。 |
| **`logs/`** | **监控分析**。记录 Loss 曲线、RMSE 误差、显存占用及训练速度，是判断模型是否收敛的核心依据。 |
| **`plots/`** | **可视化**。存放脚本自动生成的训练结果图表。 |
| **`final_models/`** | **交付部署**。存放训练完成的成品模型。可结合 `scripts/lmp_model_convert.py` 进行格式转换，用于 **LAMMPS** 生产级 MD 模拟。 |

### 模型构建具体流程
本节将结合标准项目结构，以DMC(碳酸二甲酯)溶剂力场开发为例，介绍从数据准备到结果分析的全流程。

1. 数据准备 (Data Preparation)
    模型开发的第一步是获取高质量的训练数据，OneScience平台已预置经过xTB(半经验量子力学)计算的DMC数据集（`solvent_xtb_train_200.xyz`、`solvent_xtb_test.xyz`），下载后放置在项目的data/目录下

2. 模型训练 (Model Training)
    训练脚本统一存放于 scripts/ 目录中。请在项目根目录 (MACE_ROOT) 下根据硬件资源选择合适的启动方式。
    * `单卡训练 (推荐入门)`： 适用于调试或小规模数据集。在主目录下直接运行 `bash ./scripts/train.sh`
    * `进阶训练`： scripts下还提供了多卡训练脚本（`train_multi.sh`）和多节点训练脚本（`slurm_one_node.sh`、`slurm_multi_node.sh`）用于生产级模型训练。

3. 模型测试 （Model Evaluation）
    训练完成后，使用评估工具对模型进行测试，验证其泛化能力。测试脚本`eval_configs.py`位于scripts目录下，用户可直接运行，对测试集构型进行推理，预测能量、受力及应力，并将结果写入outputs/目录下。

4. 评测标准 （Evaluation Metrics）
    模型性能主要从物理精度和计算效率两个维度进行评估：物理量精度考虑模型对能量（关注RMSE/MAE指标，meV/atom）、力（关注RMSE指标 eV/A）和应力 （可选，评估晶格应力张量的准确性），同时脚本会打印推理耗时，评估模型在当前硬件上的计算速度。

5. 结果分析 （Result Analysis）
    训练结束后，程序会自动生成可视化的分析图表，存放于 outputs/{model_name}/plots/ 目录下。对于收敛曲线，检查Loss是否随Epoch平稳下降，判断模型是否出现过拟合（训练集 Loss 降但验证集 Loss 升）或欠拟合。对于对角线图，观察预测值（Y轴）与真实值（X轴）是否紧密分布在对角线附近。分布越紧密，代表 R² 越接近 1，模型精度越高。
